# Customer Drop-off, Conversion & Attrition Risk Analysis for an E-commerce Product

---

### Objective:
#### Analyse customer engagement, identify conversion drop-off points, and segment customers by attrition risk to support product and retention decisions.
**Constraints:**  
Event-level data unavailable; analysis conducted using customer-level behavioural aggregates.

## 1. Data Overview and Quality Assessment 
---


In [1]:
# importing the data
import pandas as pd
import numpy as np
data = pd.read_csv('../data/ecommerce_customer_churn_dataset.csv')
df = pd.DataFrame(data)
#initial inspection
df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,17.9,9.0,4.0,16.3,20.8,1.0,953.33,2278.0,0,Q1
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,42.8,7.0,3.0,NaN,23.3,3.0,1067.47,3028.0,0,Q4
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0.0,4.0,1.0,NaN,8.8,NaN,1289.75,2317.0,0,Q4
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,41.4,2.0,5.0,85.9,31.0,3.0,2340.92,2674.0,0,Q1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,NaN,19.1,9.0,...,37.9,1.0,11.0,83.0,50.4,4.0,3041.29,5354.0,0,Q4


#### Inspecting the data

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            47505 non-null  float64
 1   Gender                         50000 non-null  object 
 2   Country                        50000 non-null  object 
 3   City                           50000 non-null  object 
 4   Membership_Years               50000 non-null  float64
 5   Login_Frequency                50000 non-null  float64
 6   Session_Duration_Avg           46601 non-null  float64
 7   Pages_Per_Session              47000 non-null  float64
 8   Cart_Abandonment_Rate          50000 non-null  float64
 9   Wishlist_Items                 46000 non-null  float64
 10  Total_Purchases                50000 non-null  float64
 11  Average_Order_Value            50000 non-null  float64
 12  Days_Since_Last_Purchase       47000 non-null 

In [3]:
df.shape

(50000, 25)

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,47505.0,37.802968,11.834668,5.00,29.0000,38.000,46.00,200.000000
Membership_Years,50000.0,2.984009,2.059105,0.10,1.4000,2.500,4.00,10.000000
Login_Frequency,50000.0,11.624660,7.810657,0.00,6.0000,11.000,17.00,46.000000
Session_Duration_Avg,46601.0,27.660754,10.871013,1.00,19.7000,26.800,34.70,75.600000
Pages_Per_Session,47000.0,8.737811,3.778220,1.00,6.0000,8.400,11.20,24.100000
Cart_Abandonment_Rate,50000.0,57.079973,16.282723,0.00,46.4000,58.100,68.70,143.743350
Wishlist_Items,46000.0,4.298391,3.189754,0.00,2.0000,4.000,6.00,28.000000
Total_Purchases,50000.0,13.111576,7.017312,-13.00,8.0000,12.000,17.00,128.700000
Average_Order_Value,50000.0,123.117330,175.569714,26.38,87.0500,112.970,144.44,9666.379178
Days_Since_Last_Purchase,47000.0,29.792872,29.695062,0.00,9.0000,21.000,41.00,287.000000


### Missing Data Audit

In [5]:
missing_summary = (
    df.isnull()
      .sum()
      .to_frame("missing_count")
      .assign(missing_percent=lambda x: x["missing_count"] / len(df) * 100)
      .sort_values("missing_percent", ascending=False)
)

missing_summary

,missing_count,missing_percent
Social_Media_Engagement_Score,6000,12.000
Credit_Balance,5500,11.000
Mobile_App_Usage,5000,10.000
Returns_Rate,4491,8.982
Wishlist_Items,4000,8.000
Product_Reviews_Written,3500,7.000
Discount_Usage_Rate,3500,7.000
Session_Duration_Avg,3399,6.798
Pages_Per_Session,3000,6.000
Days_Since_Last_Purchase,3000,6.000


## 2. Data Cleaning and Imputation

---
Behavioural engagement metrics were imputed using the median to reduce the influence of outliers while preserving typical user behaviour. Count-based metrics were filled with zero, reflecting the absence of recorded activity. Time-based recency values were not imputed; instead, missingness was retained via an indicator flag to avoid introducing artificial recency assumptions.

In [6]:
df['Age'] = df['Age'].fillna(value=df['Age'].median())

#### Age Data Validation

Age values outside a plausible adult customer range (below 18 or above 90) were treated as invalid and set to missing. These values likely represent data entry or synthetic generation errors rather than true customer attributes.

Invalid ages were subsequently imputed using the median to preserve the overall demographic distribution while avoiding distortion from implausible values. A flag was retained to track records affected by this adjustment.

In [7]:
#implausible age range correction
min_age = 18
max_age = 90

# Flag implausible ages
df["age_out_of_range_flag"] = (
    (df["Age"] < min_age) | (df["Age"] > max_age)
).astype(int)

# Set implausible ages to NaN
df.loc[df["age_out_of_range_flag"] == 1, "Age"] = np.nan

# Impute with median
df["Age"] = df["Age"].fillna(df["Age"].median())

#### Invalid Purchase Count Handling
Negative values were observed in the Total_Purchases field, which are not logically possible in an e-commerce context and likely represent data generation or ingestion errors. These values were treated as invalid and set to missing.

Missing purchase counts were imputed as zero to avoid fabricating purchase activity, ensuring a conservative and business-consistent treatment. A flag was retained to track records affected by this correction.

In [8]:
df["invalid_total_purchases_flag"] = (df["Total_Purchases"] < 0).astype(int)
df.loc[df["Total_Purchases"] < 0, "Total_Purchases"] = np.nan
df["Total_Purchases"] = df["Total_Purchases"].fillna(0)

#### Data type normalization

In [9]:
df["Churned"] = df["Churned"].astype(int)
df["Signup_Quarter"] = df["Signup_Quarter"].astype("category")

### Column Grouping

In [10]:
behavioral_metrics = [
    "Login_Frequency",
    "Session_Duration_Avg",
    "Pages_Per_Session",
    "Social_Media_Engagement_Score",
    "Email_Open_Rate",
    "Mobile_App_Usage"
]

count_metrics = [
    "Wishlist_Items",
    "Product_Reviews_Written",
    "Customer_Service_Calls"
]

time_based_metrics = [
    "Days_Since_Last_Purchase"
]

### Behavorial metrics
Behavioural engagement metrics with missing values were imputed using the median. This decision reflects the skewed nature of engagement data and ensures robust segmentation without overstating or understating user activity. Missing values are treated as unknown rather than zero activity.

In [11]:
#Filling each of the missing values with their corresponding medians
for col in behavioral_metrics:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    

In [12]:
#validation
#df[behavioral_metrics].isnull().sum()

### Count metrics
Count metrics with missing values were filled with 0. This choice was made as missing data in these metrics highly likely implies a lack of engagement and and are useful to the analysis.

In [13]:
for col in count_metrics:
    df[col] = df[col].fillna(0)

### Time-Based metric
This metric with the missingness needs to be preserved as any assumptions will cause distortion in the data. So a flag column will be set for missing values and the column is left untouched

In [14]:
df["missing_last_purchase_flag"] = df["Days_Since_Last_Purchase"].isnull().astype(int)

## 3. Feature Engineering

---
### Engagement Score
A composite engagement score was constructed using normalised login frequency, average session duration, and pages per session. A simple average was chosen to maintain interpretability and avoid arbitrary weighting. Users above the median engagement score were classified as “engaged” for funnel analysis.

In [15]:
from sklearn.preprocessing import MinMaxScaler

#normalising the values
engagement_cols = [
    "Login_Frequency",
    "Session_Duration_Avg",
    "Pages_Per_Session"
]

scaler = MinMaxScaler()

df[[f"{col}_norm" for col in engagement_cols]] = scaler.fit_transform(
    df[engagement_cols]
)


In [16]:
df["engagement_score"] = df[
    [f"{col}_norm" for col in engagement_cols]
].mean(axis=1)

In [17]:
engagement_threshold = df["engagement_score"].median()

df["engaged_flag"] = (df["engagement_score"] >= engagement_threshold).astype(int)

### Conversion and Value Features
Conversion was defined as at least one completed purchase. Revenue per customer was derived to support value-based segmentation. Cart abandonment was treated as a relative friction score and high-abandonment users were identified using percentile-based thresholds to avoid unit ambiguity.

In [18]:
df["purchase_flag"] = (df["Total_Purchases"] >= 1).astype(int)

#### Revenue per Customer

In [19]:
df["revenue_per_customer"] = df["Total_Purchases"] * df["Average_Order_Value"]

Revenue per customer was capped at the 99th percentile (£5,051) to limit the influence of extreme outliers likely arising from synthetic data artefacts, while preserving meaningful variation among high-value customers.

In [20]:
revenue_cap = df["revenue_per_customer"].quantile(0.99)

df["revenue_per_customer_capped"] = np.where(
    df["revenue_per_customer"] > revenue_cap,
    revenue_cap,
    df["revenue_per_customer"]
)

#### High-value Customer Flag

In [21]:
value_threshold = df["revenue_per_customer_capped"].median()

df["high_value_customer_flag"] = (
    df["revenue_per_customer_capped"] >= value_threshold
).astype(int)

#### High cart abandonment Flag

In [22]:
abandonment_threshold = df["Cart_Abandonment_Rate"].quantile(0.75)

df["high_cart_abandonment_flag"] = (
    df["Cart_Abandonment_Rate"] >= abandonment_threshold
).astype(int)

#### High Return Rate Flag

In [23]:
returns_threshold = df["Returns_Rate"].quantile(0.75)

df["high_returns_flag"] = (df["Returns_Rate"] >= returns_threshold).astype(int)

### Recency and Frequency Segmentation
Customers were segmented into recency and purchase-frequency buckets to support churn risk assessment. Thresholds were chosen to reflect common retention intervention windows rather than optimise predictive accuracy.

#### Recency bucketing

In [24]:
def recency_bucket(days):
    if pd.isnull(days):
        return "Unknown"
    elif days <= 30:
        return "Recent"
    elif days <= 90:
        return "At Risk"
    else:
        return "Dormant"

df["recency_bucket"] = df["Days_Since_Last_Purchase"].apply(recency_bucket)

#### Engagement Bucketing

A composite engagement score was bucketed into Low, Medium, and High segments using quantile-based thresholds to ensure robustness to skewed behavioural distributions.

In [25]:
df["engagement_bucket"] = pd.qcut(
    df["engagement_score"],
    q=3,
    labels=["Low", "Medium", "High"]
)

#### Purchase Frequency Bucketing

Customers were grouped into purchase-frequency segments based on cumulative purchase count. Thresholds were chosen to distinguish one-time buyers from occasional and frequent purchasers, supporting churn risk and value-based analysis.

In [26]:
max_purchases = df["Total_Purchases"].max()

df["purchase_frequency_bucket"] = pd.cut(
    df["Total_Purchases"],
    bins=[0, 1, 5, max_purchases],
    labels=["One-time", "Occasional", "Frequent"],
    include_lowest=True
)

#### Purchase Frequency Distribution Caveat

The purchase frequency distribution is heavily skewed towards frequent purchasers, which is atypical for most e-commerce platforms. This suggests the dataset may over-represent high-activity users or reflect synthetic generation assumptions. As a result, purchase frequency is interpreted as a relative intensity signal rather than a literal customer lifecycle distribution.

In [27]:
df["purchase_frequency_bucket"].value_counts()

purchase_frequency_bucket
Frequent      45455
Occasional     4360
One-time        185
Name: count, dtype: int64

#### Creating relative distribution

In [28]:
df["purchase_frequency_quantile"] = pd.qcut(
    df["Total_Purchases"],
    q=3,
    labels=["Low", "Medium", "High"]
)

In [29]:
df["purchase_frequency_quantile"].value_counts()

purchase_frequency_quantile
Low       18993
High      15877
Medium    15130
Name: count, dtype: int64

## 4. Funnel Construction
---

### Exploratory Proxy Funnel Construction

Due to the absence of event-level data, a customer-level proxy funnel was constructed using behavioural engagement, friction indicators, and purchase outcomes. This approach allows identification of meaningful drop-off points while remaining robust to data granularity constraints.

Note: This funnel was constructed as an initial proxy but exhibited near-total conversion between intent and purchase, indicating structural bias in the dataset.

In [30]:
df["stage_engaged"] = df["engaged_flag"]

In [31]:
df["stage_purchase_intent"] = (
    (df["stage_engaged"] == 1) &
    (df["high_cart_abandonment_flag"] == 0)
).astype(int)

In [32]:
df["stage_converted"] = (
    (df["stage_purchase_intent"] == 1) &
    (df["purchase_flag"] == 1)
).astype(int)

In [33]:
df["stage_retained"] = (
    (df["stage_converted"] == 1) &
    (df["Churned"] == 0)
).astype(int)

df["stage_churned"] = (
    (df["stage_converted"] == 1) &
    (df["Churned"] == 1)
).astype(int)

#### Funnel Summary Table

In [34]:
funnel_counts = {
    "Engaged Users": df["stage_engaged"].sum(),
    "Purchase Intent Users": df["stage_purchase_intent"].sum(),
    "Converted Users": df["stage_converted"].sum(),
    "Retained Users": df["stage_retained"].sum()
}

funnel_df = pd.DataFrame.from_dict(
    funnel_counts,
    orient="index",
    columns=["user_count"]
)

funnel_df

,user_count
Engaged Users,25000
Purchase Intent Users,23426
Converted Users,23410
Retained Users,18789


In [35]:
funnel_df["conversion_rate"] = (
    funnel_df["user_count"] /
    funnel_df["user_count"].shift(1)
)

funnel_df

,user_count,conversion_rate
Engaged Users,25000,NaN
Purchase Intent Users,23426,0.937040
Converted Users,23410,0.999317
Retained Users,18789,0.802606


In [36]:
funnel_df["dropoff_rate"] = 1 - funnel_df["conversion_rate"]
funnel_df

,user_count,conversion_rate,dropoff_rate
Engaged Users,25000,NaN,NaN
Purchase Intent Users,23426,0.937040,0.062960
Converted Users,23410,0.999317,0.000683
Retained Users,18789,0.802606,0.197394


### Revised Funnel: Engagement -> Conversion -> Retention

Initial conversion logic based on lifetime purchase resulted in near-total conversion among engaged users, reflecting dataset bias toward historical purchasers. Conversion was therefore refined to represent active purchasing behaviour by incorporating recency criteria, aligning the funnel with decision-making and retention use cases.

In [37]:
df["stage_converted_strict"] = (
    (df["engaged_flag"] == 1) &
    (df["purchase_flag"] == 1) &
    (df["recency_bucket"].isin(["Recent", "At Risk"]))
).astype(int)

In [38]:
funnel_counts_strict = {
    "Engaged Users": (df["engaged_flag"] == 1).sum(),

    "Active Converted Users": (
        df["stage_converted_strict"] == 1
    ).sum(),

    "Retained Users": (
        (df["stage_converted_strict"] == 1) &
        (df["Churned"] == 0)
    ).sum()
}

funnel_strict_df = pd.DataFrame.from_dict(
    funnel_counts_strict,
    orient="index",
    columns=["user_count"]
)

funnel_strict_df["conversion_rate"] = (
    funnel_strict_df["user_count"] /
    funnel_strict_df["user_count"].shift(1)
)

funnel_strict_df["dropoff_rate"] = 1 - funnel_strict_df["conversion_rate"]

funnel_strict_df

,user_count,conversion_rate,dropoff_rate
Engaged Users,25000,NaN,NaN
Active Converted Users,22405,0.89620,0.10380
Retained Users,17965,0.80183,0.19817


## 5. Hierarchical Rule-Based Churn Segmentation
---

Initial additive risk scoring produced non-monotonic churn rates due to the dominant influence of recency. To preserve interpretability and align with behavioural churn dynamics, recency was treated as the primary risk gate, with engagement, value, and friction signals used as secondary escalation factors.

Churn analysis restricted to engaged and active users to avoid contaminating insights with inactive or irrelevant customers 

In [39]:
churn_base = df[df["stage_converted_strict"] == 1].copy()

churn_base.shape

(22405, 49)

In [40]:
churn_risk_base = churn_base[churn_base["Churned"] == 0].copy()
churn_risk_base.shape

(17965, 49)

#### Base Risk Tier - Recency Risk

In [41]:
churn_risk_base["base_risk"] = churn_risk_base["recency_bucket"].map({
    "Recent": "Low",
    "At Risk": "Medium",
    "Dormant": "High"
})

#### Secondary Risk Tier
##### Engagement Risk

In [42]:
churn_risk_base["engagement_risk"] = df["engagement_bucket"].map({
    "High": "Low",
    "Medium": "Medium",
    "Low": "High"
})

##### Lifetime Value Risk

In [43]:
churn_risk_base["value_bucket"] = pd.qcut(
    churn_risk_base["Lifetime_Value"],
    q=3,
    labels=["Low", "Medium", "High"]
)

churn_risk_base["value_risk"] = churn_risk_base["value_bucket"].map({
    "High": "Low",
    "Medium": "Medium",
    "Low": "High"
})

##### Friction Risk

In [44]:
churn_risk_base["friction_risk"] = np.where(
    (churn_risk_base["high_cart_abandonment_flag"] == 1) |
    (churn_risk_base["high_returns_flag"] == 1),
    "High",
    "Low"
)

#### Churn Risk Score

In [45]:
risk_map = {"Low": 0, "Medium": 1, "High": 2}

churn_risk_base["secondary_risk_score"] = (
    churn_risk_base["engagement_risk"].map(risk_map).astype(int) +
    churn_risk_base["value_risk"].map(risk_map).astype(int) +
    churn_risk_base["friction_risk"].map(risk_map).astype(int)
)

In [46]:
churn_risk_base["secondary_risk_score"].describe()

count    17965.000000
mean         1.881659
std          1.447293
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          5.000000
Name: secondary_risk_score, dtype: float64

In [47]:
def assign_churn_risk(row):
    if row["base_risk"] == "High":
        return "High Risk"

    if row["base_risk"] == "Medium" and row["secondary_risk_score"] >= 3:
        return "High Risk"

    if row["base_risk"] == "Medium":
        return "Medium Risk"

    if row["base_risk"] == "Low" and row["secondary_risk_score"] >= 4:
        return "Medium Risk"

    return "Low Risk"


churn_risk_base["churn_risk_segment"] = churn_risk_base.apply(
    assign_churn_risk,
    axis=1
)


## 6. Churn Risk Segment Validation (Internal Consistency)
---

In [51]:
churn_risk_base.groupby("churn_risk_segment").agg(
    avg_days_since_last_purchase=("Days_Since_Last_Purchase", "mean"),
    avg_engagement=("engagement_score", "mean"),
    avg_ltv=("Lifetime_Value", "mean"),
    high_friction_rate=("friction_risk", lambda x: (x == "High").mean())
)

,avg_days_since_last_purchase,avg_engagement,avg_ltv,high_friction_rate
churn_risk_segment,,,,
High Risk,50.272112,0.363027,1172.436108,0.634755
Low Risk,12.715923,0.439811,1889.488812,0.162123
Medium Risk,39.560403,0.426833,1773.573071,0.362769


High-risk customers exhibit the longest purchase inactivity, lowest engagement, highest friction incidence, and reduced lifetime value, validating the segmentation’s internal consistency.

## 7. Key Insights
---

### 1. Retention is the primary revenue leakage point
Although ~90% of engaged users convert to active purchasers, ~20% of active converted users fall into high churn risk, indicating that post-purchase retention, rather than acquisition or conversion, represents the largest revenue leakage point.  
**Source:** Funnel + churn segmentation.
### 2. Recency dominates churn risk
High-risk customers last purchased on average ~50 days ago, nearly 4× longer than low-risk customers, confirming recency as the strongest early indicator of churn risk.  
**Source:** Validation table
### 3. High-risk customers are historically valuable
Despite elevated churn risk, high-risk customers retain substantial historical lifetime value (~1,170), indicating that churn risk is driven by disengagement and friction rather than inherent low value.
### 4. Friction is a critical churn escalator
Over 63% of high-risk customers exhibit high friction behaviour (cart abandonment or high returns), compared to only ~16% of low-risk customers, highlighting friction reduction as a high-impact retention lever.
### 5. Medium-risk customers offer highest ROI
Medium-risk customers show moderate disengagement and friction while maintaining high average lifetime value (~1,770), making them the most cost-effective segment for proactive retention interventions.
### 6. Engagement metrics require recency context
Engagement scores decline across risk tiers but do not collapse entirely in the high-risk segment, suggesting that engagement metrics without recency context may overestimate customer health.  
**Source:** As highlighted in Section 5

## 8. Business Implications & Next Steps
---

### 1. Retention Efforts Should Prioritise Medium-Risk Customers
As shown in the churn risk segmentation and validation (Sections 5 and 6), medium-risk customers retain high lifetime value while exhibiting early signs of disengagement and friction. Targeted retention interventions for this group are likely to yield the highest return on investment.

### 2. Recency-Based Triggers Should Drive Retention Campaigns
The dominance of purchase recency in differentiating churn risk suggests that time-since-last-purchase thresholds can be used as simple, high-signal triggers for re-engagement campaigns, without requiring complex predictive models.

### 3. Friction Reduction Is a Key Lever for High-Risk Customers
High friction incidence among high-risk customers (Section 6) indicates that operational improvements—such as checkout optimisation or returns policy adjustments—may be more effective than promotional incentives alone.

### 4. Engagement Metrics Require Contextual Interpretation
While engagement scores decline across risk tiers, they remain non-zero even among high-risk customers, suggesting that engagement metrics without recency context may overestimate customer health and should be interpreted cautiously.

### 5. Data & Modelling Extensions
Access to event-level interaction data and temporal purchase histories would enable the development of predictive churn models and causal retention experiments. However, this analysis demonstrates that decision-ready insights can be derived from aggregated data through transparent assumptions and rule-based segmentation.